In [11]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# Load cấu hình
load_dotenv()
PG_HOST     = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT     = os.getenv("POSTGRES_PORT", "5432")
PG_DB       = os.getenv("POSTGRES_DB", "paysim_dw")
PG_USER     = os.getenv("POSTGRES_USER", "paysim")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "paysim123")

DATABASE_URL = f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"

W = 75  # Output width
engine = create_engine(DATABASE_URL)

In [8]:
def fetch(query: str) -> pd.DataFrame:
    """Run an SQL query and return a DataFrame."""
    try:
        return pd.read_sql_query(query, DATABASE_URL)
    except Exception as e:
        print(f"  [ERROR] Query failed: {e}")
        return pd.DataFrame()

In [25]:
df_recent = fetch("""
        select count(*) as daily_anomalie
        from fact_binance_trades 
        where is_anomaly = True and date_key >= 20260401;
    """)
if df_recent.empty:
    print("No recent anomalies found.")
else:
    print(f"Recent Anomalies (showing {min(len(df_recent), 10)} records):")
    print(df_recent.to_string(index=False))

Recent Anomalies (showing 1 records):
 daily_anomalie
         153706


In [ ]:
try:
    with engine.begin() as conn: # engine.begin() tự động commit sau khi chạy xong
        # 1. Reset toàn bộ về False
            conn.execute(text("""SELECT 
                batch_id, 
                sink_name, 
                row_count, 
                latency_ms,
                ROUND((row_count::numeric / NULLIF(latency_ms, 0)) * 1000, 2) AS throughput_per_sec,
                recorded_at
                FROM fact_pipeline_latency
                ORDER BY recorded_at DESC
            LIMIT 10;
        """))
            
except Exception as e:
    print(f"Có lỗi xảy ra: {e}")

In [21]:
try:
    with engine.begin() as conn: # engine.begin() tự động commit sau khi chạy xong
        # 1. Reset toàn bộ về False
        conn.execute(text("UPDATE fact_binance_trades  SET is_anomaly = False WHERE date_key >= 20260401;"))
        conn.execute(text("UPDATE fact_binance_trades  SET is_anomaly = True WHERE (amount_usd > 100000 or z_score > 3) and date_key >= 20260401;"))
        print("Đã reset toàn bộ is_anomaly về False.")
        print("Đã cập nhật lại Anomaly cho các giao dịch lớn.")
        
    print("Sửa dữ liệu thành công! Hãy vào PowerBI nhấn Refresh.")
except Exception as e:
    print(f"Có lỗi xảy ra: {e}")

Đã reset toàn bộ is_anomaly về False.
Đã cập nhật lại Anomaly cho các giao dịch lớn.
Sửa dữ liệu thành công! Hãy vào PowerBI nhấn Refresh.
